In [ ]:
import numpy as np
import pandas as pd
import os
from scipy.stats import sem
import scipy

## Configuration

In [ ]:
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"] # DTD | EuroSAT | GTSRB | MNIST | RESISC45 | Stanford_Cars | SUN397 | SVHN
domain = "Base_Fine_Tuned" # Base_Fine_Tuned | Fine_Tuned_Layer_Skipping
domain_type = "Long_Training"
model_name = "CLIP_ViT_Vision" # DeiT | CLIP_ViT_Vision | Google_ViT
transformation = ["Standard", "Base_Fine_Tuned_Classifier", "Base_Linear_Probe"] # "Standard" | "Base_Fine_Tuned_Classifier" | "Base_Linear_Probe"
indices = [i for i in range(12)]
size = [i for i in range(1,6)]

## Loading Data

In [ ]:
def find_best_acc(data):
    acc = {i: [] for i in indices} # Indices, then len(data)
    for i in indices:
        for j in range(len(data)):
            acc[i].append(data[j]["Classification_Accuracy"][i])

    best_acc = {}

    for i in acc:
        arr = acc[i]
        max_val = max(arr)
        indice = arr.index(max_val)
        num_img = data[indice]["Train_Data_Size"][i][0]
        best_acc[i] = (max_val, num_img)

    final = []
    num_img = []
    for i in indices:
        final.append(best_acc[i][0])
        num_img.append(best_acc[i][1])

    best_accuracy = max(final)
    index = final.index(best_accuracy)
    best_accuracy_num_images = num_img[index]

    return best_accuracy, best_accuracy_num_images, index

In [ ]:
for k in range(len(dataset_name)):
    said = k
    results_path = f"../Data/{domain_type}/{dataset_name[said]}_{domain}" # /Entire_Transformation_Matrix_W"

    results = {}

    for i in size:
        results[i] = []
        path = f"{results_path}/{i}/Entire_Transformation_Matrix_W"
        try:
            for filename in os.listdir(path):
                if filename in [".DS_Store", f"Base_Fine_Tuned_Classifier_Results_{i}.json", f"Base_Linear_Probe_Results_{i}.json", ".ipynb_checkpoints"]: 
                    continue
                file_path = os.path.join(path, filename)
                if os.path.isfile(file_path):
                    results[i].append(file_path)
        except FileNotFoundError:
            print(f"Error: The Folder '{path}' was not found.")
        except Exception as e:
            print(f"An error occured: {e}")

        data = [pd.read_json(i) for i in results[i]]
        # data = sorted(data, key=lambda df: df["Train_Data_Size"][0])
        results[i] = data
    
    best_acc = []
    num_img = []
    index = []
    print(f"{model_name} - {dataset_name[said]}: {domain}")
    for i in size:
        acc, img, ind = find_best_acc(results[i])
        best_acc.append(acc)
        num_img.append(img)
        index.append(ind)
        print(f"Set {i} Best Accuracy: {best_acc[i-1]} | Number of training images: {num_img[i-1]} | Transformation Layer (0-11): {index[i-1]}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(best_acc)-1, loc=np.mean(best_acc), scale=sem(best_acc))
    print(f"Average: {np.mean(best_acc)}, Error: +- {ci_high-np.mean(best_acc)}, 95% Confidence Interval: {(ci_low, ci_high)}")

In [ ]:
DTD = [0.7617021276595745, 0.7755319148936171, 0.7909574468085107, 0.7813829787234042, 0.7648936170212766]
DTD_linear_probe = [0.774468085106383, 0.7734042553191489, 0.7728723404255319, 0.7728723404255319, 0.7670212765957447]
EuroSAT = [0.9874074074074074, 0.9777777777777777, 0.9885185185185185, 0.9818518518518519, 0.9796296296296296]
EuroSAT_linear_probe = [0.9614814814814815, 0.96, 0.9581481481481482, 0.9585185185185185, 0.9596296296296296]
GTSRB = [0.9874901029295329, 0.9869358669833729, 0.9882818685669041, 0.9893111638954869, 0.9876484560570071]
GTSRB_linear_probe = [0.8669041963578781, 0.8684085510688836, 0.8691211401425178, 0.8665874901029296, 0.8699129057798891]
MNIST = [0.9952, 0.9928, 0.9949, 0.995, 0.9951]
MNIST_linear_probe = [0.9879, 0.988, 0.9879, 0.9871, 0.987]
RESISC45 = [0.923968253968254, 0.9157142857142857, 0.9265079365079365, 0.9255555555555556, 0.9277777777777778]
RESISC45_linear_probe = [0.9152380952380952, 0.9165079365079365, 0.9185714285714286, 0.9176190476190477, 0.9193650793650794]
Stanford_Cars = [0.8271359283671185, 0.8307424449695312, 0.8307424449695312,  0.8297475438378311,  0.8206690710110682]
Stanford_Cars_linear_probe = [0.80089541101853, 0.7976619823405049, 0.8023877627160801, 0.79940305932098, 0.7990299713965925]
SUN397 = [0.7438790931989925,0.7422166246851385, 0.7455919395465995, 0.7470025188916877, 0.7502770780856424]
SUN397_linear_probe = [0.7364231738035264, 0.7398488664987406, 0.7423173803526448, 0.7401511335012595, 0.7350629722921914]
SVHN = [0.9629686539643516, 0.9636985248924401, 0.9627765826674862, 0.9642363245236631, 0.9670789797172711]
SVHN_linear_probe = [0.6686770129071912, 0.665680700676091, 0.66279963122311, 0.6684849416103258,0.6671020282728949]

In [ ]:
data = [DTD, EuroSAT, GTSRB, MNIST, RESISC45, Stanford_Cars, SUN397, SVHN]
data_linear_probe = [DTD_linear_probe, EuroSAT_linear_probe, GTSRB_linear_probe, MNIST_linear_probe, RESISC45_linear_probe, Stanford_Cars_linear_probe, SUN397_linear_probe, SVHN_linear_probe]

In [ ]:
for f_t, probe in zip(data, data_linear_probe):
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(f_t)-1, loc=np.mean(f_t), scale=sem(f_t))
    print(f"Fine-Tuned Average: {np.mean(f_t)}, Error: +- {ci_high-np.mean(f_t)}, 95% Confidence Interval: {(ci_low, ci_high)}")
    ci_low, ci_high = scipy.stats.t.interval(0.95, df=len(probe)-1, loc=np.mean(probe), scale=sem(probe))
    print(f"Linear Probe Average: {np.mean(probe)}, Error: +- {ci_high-np.mean(probe)}, 95% Confidence Interval: {(ci_low, ci_high)}")